In [1]:
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pickle

### d=600, n=500

In [2]:
## Homoscedastic case
d = 600
n = 500
with pd.ExcelWriter('./Results/Cirsym_Full_Results1.xlsx') as writer:
    for i in range(4):
        if i == 0:
            ## x0
            x = np.zeros((d,))
            x[0] = 1
        if i == 1:
            ## x1
            x = np.zeros((d,))
            x[0] = 1
            x[1] = 1/2
            x[2] = 1/4
            x[6] = 1/2
            x[7] = 1/8
        if i == 2:
            ## x2
            x = np.zeros((d,))
            x[99] = 1
        if i == 3:
            ## x3
            x = 1/np.linspace(1, d, d)**2
        for k in range(3):
            if k == 0:
                s_beta = 5
                beta_0 = np.zeros((d,))
                beta_0[:s_beta] = 1
            if k == 1:
                beta_0 = 1/np.sqrt(np.linspace(1, d, d))
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
            if k == 2:
                beta_0 = 1/np.linspace(1, d, d)
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
    
            # True regression function
            m_true = np.dot(x, beta_0)
            
            debl_res1 = pd.read_csv('./Results/debl_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            lproj_res1 = pd.read_csv('./Results/lproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            refit_res1 = pd.read_csv('./Results/refit_Cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            deb_est_1se = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_1se_x'+str(i)+'_beta'+str(k)+'.csv')
            if (i == 0) or (i == 2):
                rproj_res1 = pd.read_csv('./Results/rproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
                
            # Bias Comparisons
            ## Debiased lasso (Javanmard and Montarani, 2014)
            debl_obs1 = np.mean(abs(debl_res1['m_obs1'] - m_true))
            debl_obs_std1 = np.std(abs(debl_res1['m_obs1'] - m_true))
            # Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', debl_obs1, debl_obs_std1]
            Bias1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                  'Avg bias': [debl_obs1], 'Std': [debl_obs_std1]})
            
            debl_ipw1 = np.mean(abs(debl_res1['m_ipw1'] - m_true))
            debl_ipw_std1 = np.std(abs(debl_res1['m_ipw1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', debl_ipw1, debl_ipw_std1]
            
            debl_obs2 = np.mean(abs(debl_res1['m_obs2'] - m_true))
            debl_obs_std2 = np.std(abs(debl_res1['m_obs2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', debl_obs2, debl_obs_std2]
            
            debl_ipw2 = np.mean(abs(debl_res1['m_ipw2'] - m_true))
            debl_ipw_std2 = np.std(abs(debl_res1['m_ipw2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', debl_ipw2, debl_ipw_std2]
            
            debl_full = np.mean(abs(debl_res1['m_full'] - m_true))
            debl_full_std = np.std(abs(debl_res1['m_full'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', debl_full, debl_full_std]
            
            ## Debiased lasso (van de geer et al., 2014)
            lproj_obs1 = np.mean(abs(lproj_res1['m_obs1'] - m_true))
            lproj_obs_std1 = np.std(abs(lproj_res1['m_obs1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', lproj_obs1, lproj_obs_std1]

            lproj_ipw1 = np.mean(abs(lproj_res1['m_ipw1'] - m_true))
            lproj_ipw_std1 = np.std(abs(lproj_res1['m_ipw1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', lproj_ipw1, lproj_ipw_std1]

            lproj_obs2 = np.mean(abs(lproj_res1['m_obs2'] - m_true))
            lproj_obs_std2 = np.std(abs(lproj_res1['m_obs2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', lproj_obs2, lproj_obs_std2]

            lproj_ipw2 = np.mean(abs(lproj_res1['m_ipw2'] - m_true))
            lproj_ipw_std2 = np.std(abs(lproj_res1['m_ipw2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', lproj_ipw2, lproj_ipw_std2]

            lproj_full = np.mean(abs(lproj_res1['m_full'] - m_true))
            lproj_full_std = np.std(abs(lproj_res1['m_full'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', lproj_full, lproj_full_std]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                rproj_obs1 = np.mean(abs(rproj_res1['m_obs1'] - m_true))
                rproj_obs_std1 = np.std(abs(rproj_res1['m_obs1'] - m_true))
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, Observed)', rproj_obs1, rproj_obs_std1]
                
                rproj_ipw1 = np.mean(abs(rproj_res1['m_ipw1'] - m_true))
                rproj_ipw_std1 = np.std(abs(rproj_res1['m_ipw1'] - m_true))
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, IPW)', rproj_ipw1, rproj_ipw_std1]
                
                rproj_obs2 = np.mean(abs(rproj_res1['m_obs2'] - m_true))
                rproj_obs_std2 = np.std(abs(rproj_res1['m_obs2'] - m_true))
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, Observed)', rproj_obs2, rproj_obs_std2]
                
                rproj_ipw2 = np.mean(abs(rproj_res1['m_ipw2'] - m_true))
                rproj_ipw_std2 = np.std(abs(rproj_res1['m_ipw2'] - m_true))
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, IPW)', rproj_ipw2, rproj_ipw_std2]
                
                rproj_full = np.mean(abs(rproj_res1['m_full'] - m_true))
                rproj_full_std = np.std(abs(rproj_res1['m_full'] - m_true))
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (Full oracle data)', rproj_full, rproj_full_std]
            
            ## Lasso Refitting
            refit_obs1 = np.mean(abs(refit_res1['m_obs1'] - m_true))
            refit_obs_std1 = np.std(abs(refit_res1['m_obs1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, Observed)', refit_obs1, refit_obs_std1]
            
            refit_ipw1 = np.mean(abs(refit_res1['m_ipw1'] - m_true))
            refit_ipw_std1 = np.std(abs(refit_res1['m_ipw1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, IPW)', refit_ipw1, refit_ipw_std1]
            
            refit_obs2 = np.mean(abs(refit_res1['m_obs2'] - m_true))
            refit_obs_std2 = np.std(abs(refit_res1['m_obs2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, Observed)', refit_obs2, refit_obs_std2]
            
            refit_ipw2 = np.mean(abs(refit_res1['m_ipw2'] - m_true))
            refit_ipw_std2 = np.std(abs(refit_res1['m_ipw2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, IPW)', refit_ipw2, refit_ipw_std2]
            
            refit_full = np.mean(abs(refit_res1['m_full'] - m_true))
            refit_full_std = np.std(abs(refit_res1['m_full'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (Full oracle data)', refit_full, refit_full_std]
            
            ## Our proposed debiasing framework
            m1 = np.mean(abs(deb_est_1se['m_deb1'] - m_true))
            m_std1 = np.std(abs(deb_est_1se['m_deb1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, 1SE)', m1, m_std1]
            
            m2 = np.mean(abs(deb_est_1se['m_deb2'] - m_true))
            m_std2 = np.std(abs(deb_est_1se['m_deb2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, 1SE)', m2, m_std2]
            
            
            # Coverage Probability
            ## Debiased lasso (Javanmard and Montarani, 2014)
            cov_prob = (debl_res1['m_obs1'] + debl_res1['ci_len_obs1']/2 >= m_true) & \
            (debl_res1['m_obs1'] - debl_res1['ci_len_obs1']/2 <= m_true)
            # Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            Cov_prob1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                      'Coverage Probability': [np.mean(cov_prob)], 
                                      'Std': [np.std(cov_prob)]})
            
            cov_prob = (debl_res1['m_ipw1'] + debl_res1['ci_len_ipw1']/2 >= m_true) & \
            (debl_res1['m_ipw1'] - debl_res1['ci_len_ipw1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (debl_res1['m_obs2'] + debl_res1['ci_len_obs2']/2 >= m_true) & \
            (debl_res1['m_obs2'] - debl_res1['ci_len_obs2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (debl_res1['m_ipw2'] + debl_res1['ci_len_ipw2']/2 >= m_true) & \
            (debl_res1['m_ipw2'] - debl_res1['ci_len_ipw2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (debl_res1['m_full'] + debl_res1['ci_len_full']/2 >= m_true) & \
            (debl_res1['m_full'] - debl_res1['ci_len_full']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (Full data)', np.mean(cov_prob), np.std(cov_prob)]
            
            
            ## Debiased lasso (van de geer et al., 2014)
            cov_prob = (lproj_res1['m_obs1'] + np.sqrt(lproj_res1['asym_se_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_obs1'] - np.sqrt(lproj_res1['asym_se_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]

            cov_prob = (lproj_res1['m_ipw1'] + np.sqrt(lproj_res1['asym_se_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_ipw1'] - np.sqrt(lproj_res1['asym_se_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]

            cov_prob = (lproj_res1['m_obs2'] + np.sqrt(lproj_res1['asym_se_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_obs2'] - np.sqrt(lproj_res1['asym_se_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]

            cov_prob = (lproj_res1['m_ipw2'] + np.sqrt(lproj_res1['asym_se_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_ipw2'] - np.sqrt(lproj_res1['asym_se_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]

            cov_prob = (lproj_res1['m_full'] + np.sqrt(lproj_res1['asym_se_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_full'] - np.sqrt(lproj_res1['asym_se_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Full data)', np.mean(cov_prob), np.std(cov_prob)]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                cov_prob = (rproj_res1['m_obs1'] + rproj_res1['ci_len_obs1']/2 >= m_true) & \
                (rproj_res1['m_obs1'] - rproj_res1['ci_len_obs1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob)]
                
                cov_prob = (rproj_res1['m_ipw1'] + rproj_res1['ci_len_ipw1']/2 >= m_true) & \
                (rproj_res1['m_ipw1'] - rproj_res1['ci_len_ipw1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob)]
                
                cov_prob = (rproj_res1['m_obs2'] + rproj_res1['ci_len_obs2']/2 >= m_true) & \
                (rproj_res1['m_obs2'] - rproj_res1['ci_len_obs2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob)]
                
                cov_prob = (rproj_res1['m_ipw2'] + rproj_res1['ci_len_ipw2']/2 >= m_true) & \
                (rproj_res1['m_ipw2'] - rproj_res1['ci_len_ipw2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob)]
                
                cov_prob = (rproj_res1['m_full'] + rproj_res1['ci_len_full']/2 >= m_true) & \
                (rproj_res1['m_full'] - rproj_res1['ci_len_full']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (Full data)', np.mean(cov_prob), 
                                                       np.std(cov_prob)]
            
            ## Lasso refitting
            cov_prob = (refit_res1['m_obs1'] + refit_res1['ci_len_obs1']/2 >= m_true) & \
            (refit_res1['m_obs1'] - refit_res1['ci_len_obs1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_ipw1'] + refit_res1['ci_len_ipw1']/2 >= m_true) & \
            (refit_res1['m_ipw1'] - refit_res1['ci_len_ipw1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_obs2'] + refit_res1['ci_len_obs2']/2 >= m_true) & \
            (refit_res1['m_obs2'] - refit_res1['ci_len_obs2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_ipw2'] + refit_res1['ci_len_ipw2']/2 >= m_true) & \
            (refit_res1['m_ipw2'] - refit_res1['ci_len_ipw2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_full'] + refit_res1['ci_len_full']/2 >= m_true) & \
            (refit_res1['m_full'] - refit_res1['ci_len_full']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (Full data)', np.mean(cov_prob), np.std(cov_prob)]
            
            # Proposed framework        
            cov_prob = (deb_est_1se['m_deb1'] + deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_1se['m_deb1'] - deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, 1SE)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (deb_est_1se['m_deb2'] + deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_1se['m_deb2'] - deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, 1SE)', np.mean(cov_prob), np.std(cov_prob)]
            
            
            # Lengths of confidence intervals
            ## Debiased lasso (Javanmard and Montarani, 2014)
            CI_len1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                            'CI length': [np.mean(debl_res1['ci_len_obs1'])], 
                            'Std': [np.std(debl_res1['ci_len_obs1'])]})
    #         CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(debl_res1['ci_len_obs1']), 
    #                                            np.std(debl_res1['ci_len_obs1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(debl_res1['ci_len_ipw1']), 
                                               np.std(debl_res1['ci_len_ipw1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(debl_res1['ci_len_obs2']), 
                                               np.std(debl_res1['ci_len_obs2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(debl_res1['ci_len_ipw2']), 
                                               np.std(debl_res1['ci_len_ipw2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', np.mean(debl_res1['ci_len_full']), 
                                               np.std(debl_res1['ci_len_full'])]
            
            ## Debiased lasso (van de geer et al., 2014)
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_se_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_se_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_se_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_se_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_se_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_se_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_se_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_se_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_se_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_se_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2))]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, Observed)', 
                                                   np.mean(rproj_res1['ci_len_obs1']), 
                                                   np.std(rproj_res1['ci_len_obs1'])]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(rproj_res1['ci_len_ipw1']), 
                                                   np.std(rproj_res1['ci_len_ipw1'])]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, Observed)', np.mean(rproj_res1['ci_len_obs2']), 
                                                   np.std(rproj_res1['ci_len_obs2'])]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, IPW)', np.mean(rproj_res1['ci_len_ipw2']), 
                                                   np.std(rproj_res1['ci_len_ipw2'])]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (Full oracle data)', np.mean(rproj_res1['ci_len_full']), 
                                                   np.std(rproj_res1['ci_len_full'])]
            
            ## Lasso refitting
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(refit_res1['ci_len_obs1']), 
                                               np.std(refit_res1['ci_len_obs1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(refit_res1['ci_len_ipw1']), 
                                               np.std(refit_res1['ci_len_ipw1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(refit_res1['ci_len_obs2']), 
                                               np.std(refit_res1['ci_len_obs2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(refit_res1['ci_len_ipw2']), 
                                               np.std(refit_res1['ci_len_ipw2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (Full oracle data)', np.mean(refit_res1['ci_len_full']), 
                                               np.std(refit_res1['ci_len_full'])]
            
            ## Proposed debiasing framework
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, 1SE)', 
                                               np.mean(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))]
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, 1SE)', 
                                               np.mean(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))]
            
            full_res1 = pd.concat([Bias1, Cov_prob1[['Coverage Probability', 'Std']], CI_len1[['CI length', 'Std']]], axis=1)
            
            # full_res1.to_csv('./Results/Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv', index=False)
            full_res1.to_excel(writer, sheet_name='Cirsym_cov_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k), index=False)

### d=1000, n=900 (Gaussian noise)

In [6]:
## Homoscedastic case (Gaussian noises)
d = 1000
n = 900
with pd.ExcelWriter('./Results/Cirsym_Full_Results_d'+str(d)+'_n'+str(n)+'_Gaussian_noise.xlsx') as writer:
    for i in range(6):
        if i == 0:
            ## x0
            x = np.zeros((d,))
            x[0] = 1
        if i == 1:
            ## x1
            x = np.zeros((d,))
            x[0] = 1
            x[1] = 1/2
            x[2] = 1/4
            x[6] = 1/2
            x[7] = 1/8
        if i == 2:
            ## x2
            x = np.zeros((d,))
            x[99] = 1
        if i == 3:
            ## x3
            x = 1/np.linspace(1, d, d)
        if i == 4:
            ## x4
            x = 1/np.linspace(1, d, d)**2
        if i == 5:
            ## x5
            x = np.ones((d,))/np.sqrt(d)
        for k in range(3):
            if k == 0:
                s_beta = 5
                beta_0 = np.zeros((d,))
                beta_0[:s_beta] = np.sqrt(5)
            if k == 1:
                beta_0 = 1/np.sqrt(np.linspace(1, d, d))
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
            if k == 2:
                beta_0 = 1/np.linspace(1, d, d)
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
    
            # True regression function
            m_true = np.dot(x, beta_0)
            
            debl_res1 = pd.read_csv('./Results/debl_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            lproj_res1 = pd.read_csv('./Results/lproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            refit_res1 = pd.read_csv('./Results/refit_Cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            tian_res = pd.read_csv('./Results/Tian2024_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            ddr_res = pd.read_csv('./Results/DDR_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            hou_sas_res = pd.read_csv('./Results/Hou2023_SAS_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
            deb_est_1se = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_1se.csv')
            deb_est_mincv = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_mincv.csv')
            deb_est_minfeas = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_minfeas.csv')
            if (i == 0) or (i == 2):
                rproj_res1 = pd.read_csv('./Results/rproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv')
                
            # Bias Comparisons
            ## Debiased lasso (Javanmard and Montarani, 2014)
            debl_obs1 = np.mean(abs(debl_res1['m_obs1'] - m_true))
            debl_obs_std1 = np.std(abs(debl_res1['m_obs1'] - m_true))
            debl_obs_ste1 = debl_obs_std1/np.sqrt(1000)
            # Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', debl_obs1, debl_obs_std1, debl_obs_ste1]
            Bias1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                  'Avg bias': [debl_obs1], 'Std': [debl_obs_std1], 'StdErr': [debl_obs_ste1]})
            
            debl_ipw1 = np.mean(abs(debl_res1['m_ipw1'] - m_true))
            debl_ipw_std1 = np.std(abs(debl_res1['m_ipw1'] - m_true))
            debl_ipw_ste1 = debl_ipw_std1/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', debl_ipw1, 
                                           debl_ipw_std1, debl_ipw_ste1]
            
            debl_obs2 = np.mean(abs(debl_res1['m_obs2'] - m_true))
            debl_obs_std2 = np.std(abs(debl_res1['m_obs2'] - m_true))
            debl_obs_ste2 = debl_obs_std2 / np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', debl_obs2, 
                                           debl_obs_std2, debl_obs_ste2]
            
            debl_ipw2 = np.mean(abs(debl_res1['m_ipw2'] - m_true))
            debl_ipw_std2 = np.std(abs(debl_res1['m_ipw2'] - m_true))
            debl_ipw_ste2 = debl_ipw_std2/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', debl_ipw2, 
                                           debl_ipw_std2, debl_ipw_ste2]
            
            debl_full = np.mean(abs(debl_res1['m_full'] - m_true))
            debl_full_std = np.std(abs(debl_res1['m_full'] - m_true))
            debl_full_ste = debl_full_std/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', debl_full, 
                                           debl_full_std, debl_full_ste]
            
            ## Debiased lasso (van de geer et al., 2014)
            lproj_obs1 = np.mean(abs(lproj_res1['m_obs1'] - m_true))
            lproj_obs_std1 = np.std(abs(lproj_res1['m_obs1'] - m_true))
            lproj_obs_ste1 = lproj_obs_std1/np.sqrt(len(lproj_res1))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', lproj_obs1, 
                                           lproj_obs_std1, lproj_obs_ste1]

            lproj_ipw1 = np.mean(abs(lproj_res1['m_ipw1'] - m_true))
            lproj_ipw_std1 = np.std(abs(lproj_res1['m_ipw1'] - m_true))
            lproj_ipw_ste1 = lproj_ipw_std1/np.sqrt(len(lproj_res1))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', lproj_ipw1, 
                                           lproj_ipw_std1, lproj_ipw_ste1]

            lproj_obs2 = np.mean(abs(lproj_res1['m_obs2'] - m_true))
            lproj_obs_std2 = np.std(abs(lproj_res1['m_obs2'] - m_true))
            lproj_obs_ste2 = lproj_obs_std2/np.sqrt(len(lproj_res1))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', lproj_obs2, 
                                           lproj_obs_std2, lproj_obs_ste2]

            lproj_ipw2 = np.mean(abs(lproj_res1['m_ipw2'] - m_true))
            lproj_ipw_std2 = np.std(abs(lproj_res1['m_ipw2'] - m_true))
            lproj_ipw_ste2 = lproj_ipw_std2/np.sqrt(len(lproj_res1))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', lproj_ipw2, 
                                           lproj_ipw_std2, lproj_ipw_ste2]

            lproj_full = np.mean(abs(lproj_res1['m_full'] - m_true))
            lproj_full_std = np.std(abs(lproj_res1['m_full'] - m_true))
            lproj_full_ste = lproj_full_std/np.sqrt(len(lproj_res1))
            Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', lproj_full, 
                                           lproj_full_std, lproj_full_ste]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                rproj_obs1 = np.mean(abs(rproj_res1['m_obs1'] - m_true))
                rproj_obs_std1 = np.std(abs(rproj_res1['m_obs1'] - m_true))
                rproj_obs_ste1 = rproj_obs_std1/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, Observed)', rproj_obs1, rproj_obs_std1, 
                                               rproj_obs_ste1]
                
                rproj_ipw1 = np.mean(abs(rproj_res1['m_ipw1'] - m_true))
                rproj_ipw_std1 = np.std(abs(rproj_res1['m_ipw1'] - m_true))
                rproj_ipw_ste1 = rproj_ipw_std1/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, IPW)', rproj_ipw1, rproj_ipw_std1, rproj_ipw_ste1]
                
                rproj_obs2 = np.mean(abs(rproj_res1['m_obs2'] - m_true))
                rproj_obs_std2 = np.std(abs(rproj_res1['m_obs2'] - m_true))
                rproj_obs_ste2 = rproj_obs_std2/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, Observed)', rproj_obs2, rproj_obs_std2, 
                                               rproj_obs_ste2]
                
                rproj_ipw2 = np.mean(abs(rproj_res1['m_ipw2'] - m_true))
                rproj_ipw_std2 = np.std(abs(rproj_res1['m_ipw2'] - m_true))
                rproj_ipw_ste2 = rproj_ipw_std2/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, IPW)', rproj_ipw2, rproj_ipw_std2, rproj_ipw_ste2]
                
                rproj_full = np.mean(abs(rproj_res1['m_full'] - m_true))
                rproj_full_std = np.std(abs(rproj_res1['m_full'] - m_true))
                rproj_full_ste = rproj_full_std / np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Ridge projection (Full oracle data)', rproj_full, 
                                               rproj_full_std, rproj_full_ste]
            
            ## Lasso Refitting
            refit_obs1 = np.mean(abs(refit_res1['m_obs1'] - m_true))
            refit_obs_std1 = np.std(abs(refit_res1['m_obs1'] - m_true))
            refit_obs_ste1 = refit_obs_std1/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, Observed)', refit_obs1, refit_obs_std1, refit_obs_ste1]
            
            refit_ipw1 = np.mean(abs(refit_res1['m_ipw1'] - m_true))
            refit_ipw_std1 = np.std(abs(refit_res1['m_ipw1'] - m_true))
            refit_ipw_ste1 = refit_ipw_std1/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, IPW)', refit_ipw1, refit_ipw_std1, refit_ipw_ste1]
            
            refit_obs2 = np.mean(abs(refit_res1['m_obs2'] - m_true))
            refit_obs_std2 = np.std(abs(refit_res1['m_obs2'] - m_true))
            refit_obs_ste2 = refit_obs_std2/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, Observed)', refit_obs2, refit_obs_std2, refit_obs_ste2]
            
            refit_ipw2 = np.mean(abs(refit_res1['m_ipw2'] - m_true))
            refit_ipw_std2 = np.std(abs(refit_res1['m_ipw2'] - m_true))
            refit_ipw_ste2 = refit_ipw_std2/np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, IPW)', refit_ipw2, refit_ipw_std2, refit_obs_ste2]
            
            refit_full = np.mean(abs(refit_res1['m_full'] - m_true))
            refit_full_std = np.std(abs(refit_res1['m_full'] - m_true))
            refit_full_ste = refit_full_std / np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (Full oracle data)', refit_full, refit_full_std, refit_full_ste]
            
            ## Our proposed debiasing framework
            m1 = np.mean(abs(deb_est_mincv['m_deb1'] - m_true))
            m_std1 = np.std(abs(deb_est_mincv['m_deb1'] - m_true))
            m_ste1 = m_std1 /np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, Mincv)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(deb_est_mincv['m_deb2'] - m_true))
            m_std2 = np.std(abs(deb_est_mincv['m_deb2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(1000) 
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, Mincv)', m2, m_std2, m_ste2]
            
            m1 = np.mean(abs(deb_est_1se['m_deb1'] - m_true))
            m_std1 = np.std(abs(deb_est_1se['m_deb1'] - m_true))
            m_ste1 = m_std1 /np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, 1SE)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(deb_est_1se['m_deb2'] - m_true))
            m_std2 = np.std(abs(deb_est_1se['m_deb2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(1000) 
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, 1SE)', m2, m_std2, m_ste2]
            
            m1 = np.mean(abs(deb_est_minfeas['m_deb1'] - m_true))
            m_std1 = np.std(abs(deb_est_minfeas['m_deb1'] - m_true))
            m_ste1 = m_std1 /np.sqrt(1000)
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, Min-feas)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(deb_est_minfeas['m_deb2'] - m_true))
            m_std2 = np.std(abs(deb_est_minfeas['m_deb2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(1000) 
            Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, Min-feas)', m2, m_std2, m_ste2]

            ## Tian2024 AIPW
            m1 = np.mean(abs(tian_res['m_hat1'] - m_true))
            m_std1 = np.std(abs(tian_res['m_hat1'] - m_true))
            m_ste1 = m_std1 / np.sqrt(len(tian_res))
            Bias1.loc[len(Bias1.index)] = ['Tian2024 AIPW (MCAR)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(tian_res['m_hat2'] - m_true))
            m_std2 = np.std(abs(tian_res['m_hat2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(len(tian_res))
            Bias1.loc[len(Bias1.index)] = ['Tian2024 AIPW (MAR)', m2, m_std2, m_ste2]
            
            ## Chakrabortty et al. (2019) DDR
            m1 = np.mean(abs(ddr_res['m_hat_de1'] - m_true))
            m_std1 = np.std(abs(ddr_res['m_hat_de1'] - m_true))
            m_ste1 = m_std1 / np.sqrt(len(ddr_res))
            Bias1.loc[len(Bias1.index)] = ['DDR (MCAR)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(ddr_res['m_hat_de2'] - m_true))
            m_std2 = np.std(abs(ddr_res['m_hat_de2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(len(ddr_res))
            Bias1.loc[len(Bias1.index)] = ['DDR (MAR)', m2, m_std2, m_ste2]
            
            ## Hou et al. (2023) debiased SAS
            m1 = np.mean(abs(hou_sas_res['m_hat1'] - m_true))
            m_std1 = np.std(abs(hou_sas_res['m_hat1'] - m_true))
            m_ste1 = m_std1 / np.sqrt(len(hou_sas_res))
            Bias1.loc[len(Bias1.index)] = ['Hou2023 SAS (MCAR)', m1, m_std1, m_ste1]
            
            m2 = np.mean(abs(hou_sas_res['m_hat2'] - m_true))
            m_std2 = np.std(abs(hou_sas_res['m_hat2'] - m_true))
            m_ste2 = m_std2 / np.sqrt(len(hou_sas_res))
            Bias1.loc[len(Bias1.index)] = ['Hou2023 SAS (MAR)', m2, m_std2, m_ste2]
            
            
            # Coverage Probability
            ## Debiased lasso (Javanmard and Montarani, 2014)
            cov_prob = (debl_res1['m_obs1'] + debl_res1['ci_len_obs1']/2 >= m_true) & \
            (debl_res1['m_obs1'] - debl_res1['ci_len_obs1']/2 <= m_true)
            # Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            Cov_prob1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                      'Coverage Probability': [np.mean(cov_prob)], 
                                      'Std': [np.std(cov_prob)], 
                                      'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
            
            cov_prob = (debl_res1['m_ipw1'] + debl_res1['ci_len_ipw1']/2 >= m_true) & \
            (debl_res1['m_ipw1'] - debl_res1['ci_len_ipw1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (debl_res1['m_obs2'] + debl_res1['ci_len_obs2']/2 >= m_true) & \
            (debl_res1['m_obs2'] - debl_res1['ci_len_obs2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (debl_res1['m_ipw2'] + debl_res1['ci_len_ipw2']/2 >= m_true) & \
            (debl_res1['m_ipw2'] - debl_res1['ci_len_ipw2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (debl_res1['m_full'] + debl_res1['ci_len_full']/2 >= m_true) & \
            (debl_res1['m_full'] - debl_res1['ci_len_full']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (Full data)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            ## Debiased lasso (van de geer et al., 2014)
            cov_prob = (lproj_res1['m_obs1'] + np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_obs1'] - np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]

            cov_prob = (lproj_res1['m_ipw1'] + np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_ipw1'] - np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]

            cov_prob = (lproj_res1['m_obs2'] + np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_obs2'] - np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]

            cov_prob = (lproj_res1['m_ipw2'] + np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_ipw2'] - np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]

            cov_prob = (lproj_res1['m_full'] + np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (lproj_res1['m_full'] - np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Full data)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                cov_prob = (rproj_res1['m_obs1'] + rproj_res1['ci_len_obs1']/2 >= m_true) & \
                (rproj_res1['m_obs1'] - rproj_res1['ci_len_obs1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (rproj_res1['m_ipw1'] + rproj_res1['ci_len_ipw1']/2 >= m_true) & \
                (rproj_res1['m_ipw1'] - rproj_res1['ci_len_ipw1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (rproj_res1['m_obs2'] + rproj_res1['ci_len_obs2']/2 >= m_true) & \
                (rproj_res1['m_obs2'] - rproj_res1['ci_len_obs2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (rproj_res1['m_ipw2'] + rproj_res1['ci_len_ipw2']/2 >= m_true) & \
                (rproj_res1['m_ipw2'] - rproj_res1['ci_len_ipw2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (rproj_res1['m_full'] + rproj_res1['ci_len_full']/2 >= m_true) & \
                (rproj_res1['m_full'] - rproj_res1['ci_len_full']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (Full data)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            ## Lasso refitting
            cov_prob = (refit_res1['m_obs1'] + refit_res1['ci_len_obs1']/2 >= m_true) & \
            (refit_res1['m_obs1'] - refit_res1['ci_len_obs1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (refit_res1['m_ipw1'] + refit_res1['ci_len_ipw1']/2 >= m_true) & \
            (refit_res1['m_ipw1'] - refit_res1['ci_len_ipw1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (refit_res1['m_obs2'] + refit_res1['ci_len_obs2']/2 >= m_true) & \
            (refit_res1['m_obs2'] - refit_res1['ci_len_obs2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (refit_res1['m_ipw2'] + refit_res1['ci_len_ipw2']/2 >= m_true) & \
            (refit_res1['m_ipw2'] - refit_res1['ci_len_ipw2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (refit_res1['m_full'] + refit_res1['ci_len_full']/2 >= m_true) & \
            (refit_res1['m_full'] - refit_res1['ci_len_full']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (Full data)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            # Proposed framework   
            cov_prob = (deb_est_mincv['m_deb1'] + deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_mincv['m_deb1'] - deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, Mincv)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (deb_est_mincv['m_deb2'] + deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_mincv['m_deb2'] - deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, Mincv)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (deb_est_1se['m_deb1'] + deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_1se['m_deb1'] - deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, 1SE)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (deb_est_1se['m_deb2'] + deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_1se['m_deb2'] - deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, 1SE)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (deb_est_minfeas['m_deb1'] + deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_minfeas['m_deb1'] - deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, Min-feas)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            cov_prob = (deb_est_minfeas['m_deb2'] + deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (deb_est_minfeas['m_deb2'] - deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, Min-feas)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
            
            ## Tian2024 AIPW
            cov_prob = (tian_res['m_hat1'] + tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (tian_res['m_hat1'] - tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Tian2024 AIPW (MCAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            cov_prob = (tian_res['m_hat2'] + tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (tian_res['m_hat2'] - tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Tian2024 AIPW (MAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            ## Chakrabortty et al. (2019) DDR
            cov_prob = (ddr_res['ci_lower1'] <= m_true) & (m_true <= ddr_res['ci_upper1'])
            Cov_prob1.loc[len(Cov_prob1.index)] = ['DDR (MCAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            cov_prob = (ddr_res['ci_lower2'] <= m_true) & (m_true <= ddr_res['ci_upper2'])
            Cov_prob1.loc[len(Cov_prob1.index)] = ['DDR (MAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            ## Hou et al. (2023) SAS
            cov_prob = (hou_sas_res['m_hat1'] + hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (hou_sas_res['m_hat1'] - hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Hou2023 SAS (MCAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
            
            cov_prob = (hou_sas_res['m_hat2'] + hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
            (hou_sas_res['m_hat2'] - hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Hou2023 SAS (MAR)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
        
            
            # Lengths of confidence intervals
            ## Debiased lasso (Javanmard and Montarani, 2014)
            CI_len1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                            'CI length': [np.mean(debl_res1['ci_len_obs1'])], 
                            'Std': [np.std(debl_res1['ci_len_obs1'])], 
                            'StdErr': [np.std(debl_res1['ci_len_obs1'])/np.sqrt(1000)]})
    #         CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(debl_res1['ci_len_obs1']), 
    #                                            np.std(debl_res1['ci_len_obs1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(debl_res1['ci_len_ipw1']), 
                                               np.std(debl_res1['ci_len_ipw1']), 
                                               np.std(debl_res1['ci_len_ipw1'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(debl_res1['ci_len_obs2']), 
                                               np.std(debl_res1['ci_len_obs2']), 
                                               np.std(debl_res1['ci_len_obs2'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(debl_res1['ci_len_ipw2']), 
                                               np.std(debl_res1['ci_len_ipw2']), 
                                               np.std(debl_res1['ci_len_ipw2'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', np.mean(debl_res1['ci_len_full']), 
                                               np.std(debl_res1['ci_len_full']), np.std(debl_res1['ci_len_full'])/np.sqrt(1000)]
            
            ## Debiased lasso (van de geer et al., 2014)
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
            CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', 
                                               np.mean(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
            
            ## Ridge projection
            if (i == 0) or (i == 2):
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, Observed)', 
                                                   np.mean(rproj_res1['ci_len_obs1']), 
                                                   np.std(rproj_res1['ci_len_obs1']), 
                                                   np.std(rproj_res1['ci_len_obs1'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(rproj_res1['ci_len_ipw1']), 
                                                   np.std(rproj_res1['ci_len_ipw1']), 
                                                   np.std(rproj_res1['ci_len_ipw1'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, Observed)', np.mean(rproj_res1['ci_len_obs2']), 
                                                   np.std(rproj_res1['ci_len_obs2']), 
                                                   np.std(rproj_res1['ci_len_obs2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, IPW)', np.mean(rproj_res1['ci_len_ipw2']), 
                                                   np.std(rproj_res1['ci_len_ipw2']), 
                                                   np.std(rproj_res1['ci_len_ipw2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (Full oracle data)', np.mean(rproj_res1['ci_len_full']), 
                                                   np.std(rproj_res1['ci_len_full']), 
                                                   np.std(rproj_res1['ci_len_full'])/np.sqrt(1000)]
            
            ## Lasso refitting
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(refit_res1['ci_len_obs1']), 
                                               np.std(refit_res1['ci_len_obs1']), 
                                               np.std(refit_res1['ci_len_obs1'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(refit_res1['ci_len_ipw1']), 
                                               np.std(refit_res1['ci_len_ipw1']), 
                                               np.std(refit_res1['ci_len_ipw1'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(refit_res1['ci_len_obs2']), 
                                               np.std(refit_res1['ci_len_obs2']), 
                                               np.std(refit_res1['ci_len_obs2'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(refit_res1['ci_len_ipw2']), 
                                               np.std(refit_res1['ci_len_ipw2']), 
                                               np.std(refit_res1['ci_len_ipw2'])/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (Full oracle data)', np.mean(refit_res1['ci_len_full']), 
                                               np.std(refit_res1['ci_len_full']), 
                                               np.std(refit_res1['ci_len_full'])/np.sqrt(1000)]
            
            ## Proposed debiasing framework
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, Mincv)', 
                                               np.mean(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, Mincv)', 
                                               np.mean(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, 1SE)', 
                                               np.mean(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, 1SE)', 
                                               np.mean(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, Min-feas)', 
                                               np.mean(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, Min-feas)', 
                                               np.mean(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
            
            ## Tian2024 AIPW
            CI_len1.loc[len(CI_len1.index)] = ['Tian2024 AIPW (MCAR)', 
                                               np.mean(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(tian_res))]
            CI_len1.loc[len(CI_len1.index)] = ['Tian2024 AIPW (MAR)', 
                                               np.mean(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(tian_res))]
            
            ## Chakrabortty et al. (2019) DDR
            CI_len1.loc[len(CI_len1.index)] = ['DDR (MCAR)', np.mean(ddr_res['ci_length1']), 
                                               np.std(ddr_res['ci_length1']), 
                                               np.std(ddr_res['ci_length1'])/np.sqrt(len(ddr_res))]
            CI_len1.loc[len(CI_len1.index)] = ['DDR (MAR)', np.mean(ddr_res['ci_length2']), 
                                               np.std(ddr_res['ci_length2']), 
                                               np.std(ddr_res['ci_length2'])/np.sqrt(len(ddr_res))]
            
            ## Hou et al. (2023) SAS
            CI_len1.loc[len(CI_len1.index)] = ['Hou2023 SAS (MCAR)', 
                                               np.mean(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(hou_sas_res))]
            CI_len1.loc[len(CI_len1.index)] = ['Hou2023 SAS (MAR)', 
                                               np.mean(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                               np.std(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(hou_sas_res))]
            
            full_res1 = pd.concat([Bias1, Cov_prob1[['Coverage Probability', 'Std', 'StdErr']], 
                                   CI_len1[['CI length', 'Std', 'StdErr']]], axis=1)
            
            # full_res1.to_csv('./Results/Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv', index=False)
            full_res1.to_excel(writer, sheet_name='Cirsym_cov_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k), index=False)

### d=1000, n=900 (other types of noises)

In [7]:
d = 1000
n = 900
# noises_type = ['laperr', 'uniferr', 'terr']
noises_type = ['laperr', 'terr']
for noise in noises_type:
    with pd.ExcelWriter('./Results/Cirsym_Full_Results_d'+str(d)+'_n'+str(n)+'_'+str(noise)+'.xlsx') as writer:
        for i in range(6):
            if i == 0:
                ## x0
                x = np.zeros((d,))
                x[0] = 1
            if i == 1:
                ## x1
                x = np.zeros((d,))
                x[0] = 1
                x[1] = 1/2
                x[2] = 1/4
                x[6] = 1/2
                x[7] = 1/8
            if i == 2:
                ## x2
                x = np.zeros((d,))
                x[99] = 1
            if i == 3:
                ## x3
                x = 1/np.linspace(1, d, d)
            if i == 4:
                ## x4
                x = 1/np.linspace(1, d, d)**2
            if i == 5:
                ## x5
                x = np.ones((d,))/np.sqrt(d)
            for k in range(3):
                if k == 0:
                    s_beta = 5
                    beta_0 = np.zeros((d,))
                    beta_0[:s_beta] = np.sqrt(5)
                if k == 1:
                    beta_0 = 1/np.sqrt(np.linspace(1, d, d))
                    beta_0 = 5*beta_0/np.linalg.norm(beta_0)
                if k == 2:
                    beta_0 = 1/np.linspace(1, d, d)
                    beta_0 = 5*beta_0/np.linalg.norm(beta_0)
        
                # True regression function
                m_true = np.dot(x, beta_0)
                
                debl_res1 = pd.read_csv('./Results/debl_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                lproj_res1 = pd.read_csv('./Results/lproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                refit_res1 = pd.read_csv('./Results/refit_Cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                tian_res = pd.read_csv('./Results/Tian2024_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                ddr_res = pd.read_csv('./Results/DDR_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                hou_sas_res = pd.read_csv('./Results/Hou2023_SAS_CirSym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                deb_est_1se = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'_1se.csv')
                deb_est_mincv = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'_mincv.csv')
                deb_est_minfeas = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'_minfeas.csv')
                if (i == 0) or (i == 2):
                    rproj_res1 = pd.read_csv('./Results/rproj_cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_'+str(noise)+'.csv')
                    
                # Bias Comparisons
                ## Debiased lasso (Javanmard and Montarani, 2014)
                debl_obs1 = np.mean(abs(debl_res1['m_obs1'] - m_true))
                debl_obs_std1 = np.std(abs(debl_res1['m_obs1'] - m_true))
                debl_obs_ste1 = debl_obs_std1/np.sqrt(1000)
                # Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', debl_obs1, debl_obs_std1, debl_obs_ste1]
                Bias1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                      'Avg bias': [debl_obs1], 'Std': [debl_obs_std1], 'StdErr': [debl_obs_ste1]})
                
                debl_ipw1 = np.mean(abs(debl_res1['m_ipw1'] - m_true))
                debl_ipw_std1 = np.std(abs(debl_res1['m_ipw1'] - m_true))
                debl_ipw_ste1 = debl_ipw_std1/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', debl_ipw1, 
                                               debl_ipw_std1, debl_ipw_ste1]
                
                debl_obs2 = np.mean(abs(debl_res1['m_obs2'] - m_true))
                debl_obs_std2 = np.std(abs(debl_res1['m_obs2'] - m_true))
                debl_obs_ste2 = debl_obs_std2 / np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', debl_obs2, 
                                               debl_obs_std2, debl_obs_ste2]
                
                debl_ipw2 = np.mean(abs(debl_res1['m_ipw2'] - m_true))
                debl_ipw_std2 = np.std(abs(debl_res1['m_ipw2'] - m_true))
                debl_ipw_ste2 = debl_ipw_std2/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', debl_ipw2, 
                                               debl_ipw_std2, debl_ipw_ste2]
                
                debl_full = np.mean(abs(debl_res1['m_full'] - m_true))
                debl_full_std = np.std(abs(debl_res1['m_full'] - m_true))
                debl_full_ste = debl_full_std/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', debl_full, 
                                               debl_full_std, debl_full_ste]
                
                ## Debiased lasso (van de geer et al., 2014)
                lproj_obs1 = np.mean(abs(lproj_res1['m_obs1'] - m_true))
                lproj_obs_std1 = np.std(abs(lproj_res1['m_obs1'] - m_true))
                lproj_obs_ste1 = lproj_obs_std1/np.sqrt(len(lproj_res1))
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', lproj_obs1, 
                                               lproj_obs_std1, lproj_obs_ste1]
    
                lproj_ipw1 = np.mean(abs(lproj_res1['m_ipw1'] - m_true))
                lproj_ipw_std1 = np.std(abs(lproj_res1['m_ipw1'] - m_true))
                lproj_ipw_ste1 = lproj_ipw_std1/np.sqrt(len(lproj_res1))
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', lproj_ipw1, 
                                               lproj_ipw_std1, lproj_ipw_ste1]
    
                lproj_obs2 = np.mean(abs(lproj_res1['m_obs2'] - m_true))
                lproj_obs_std2 = np.std(abs(lproj_res1['m_obs2'] - m_true))
                lproj_obs_ste2 = lproj_obs_std2/np.sqrt(len(lproj_res1))
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', lproj_obs2, 
                                               lproj_obs_std2, lproj_obs_ste2]
    
                lproj_ipw2 = np.mean(abs(lproj_res1['m_ipw2'] - m_true))
                lproj_ipw_std2 = np.std(abs(lproj_res1['m_ipw2'] - m_true))
                lproj_ipw_ste2 = lproj_ipw_std2/np.sqrt(len(lproj_res1))
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', lproj_ipw2, 
                                               lproj_ipw_std2, lproj_ipw_ste2]
    
                lproj_full = np.mean(abs(lproj_res1['m_full'] - m_true))
                lproj_full_std = np.std(abs(lproj_res1['m_full'] - m_true))
                lproj_full_ste = lproj_full_std/np.sqrt(len(lproj_res1))
                Bias1.loc[len(Bias1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', lproj_full, 
                                               lproj_full_std, lproj_full_ste]
                
                ## Ridge projection
                if (i == 0) or (i == 2):
                    rproj_obs1 = np.mean(abs(rproj_res1['m_obs1'] - m_true))
                    rproj_obs_std1 = np.std(abs(rproj_res1['m_obs1'] - m_true))
                    rproj_obs_ste1 = rproj_obs_std1/np.sqrt(1000)
                    Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, Observed)', rproj_obs1, rproj_obs_std1, 
                                                   rproj_obs_ste1]
                    
                    rproj_ipw1 = np.mean(abs(rproj_res1['m_ipw1'] - m_true))
                    rproj_ipw_std1 = np.std(abs(rproj_res1['m_ipw1'] - m_true))
                    rproj_ipw_ste1 = rproj_ipw_std1/np.sqrt(1000)
                    Bias1.loc[len(Bias1.index)] = ['Ridge projection (MCAR, IPW)', rproj_ipw1, rproj_ipw_std1, rproj_ipw_ste1]
                    
                    rproj_obs2 = np.mean(abs(rproj_res1['m_obs2'] - m_true))
                    rproj_obs_std2 = np.std(abs(rproj_res1['m_obs2'] - m_true))
                    rproj_obs_ste2 = rproj_obs_std2/np.sqrt(1000)
                    Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, Observed)', rproj_obs2, rproj_obs_std2, 
                                                   rproj_obs_ste2]
                    
                    rproj_ipw2 = np.mean(abs(rproj_res1['m_ipw2'] - m_true))
                    rproj_ipw_std2 = np.std(abs(rproj_res1['m_ipw2'] - m_true))
                    rproj_ipw_ste2 = rproj_ipw_std2/np.sqrt(1000)
                    Bias1.loc[len(Bias1.index)] = ['Ridge projection (MAR, IPW)', rproj_ipw2, rproj_ipw_std2, rproj_ipw_ste2]
                    
                    rproj_full = np.mean(abs(rproj_res1['m_full'] - m_true))
                    rproj_full_std = np.std(abs(rproj_res1['m_full'] - m_true))
                    rproj_full_ste = rproj_full_std / np.sqrt(1000)
                    Bias1.loc[len(Bias1.index)] = ['Ridge projection (Full oracle data)', rproj_full, 
                                                   rproj_full_std, rproj_full_ste]
                
                ## Lasso Refitting
                refit_obs1 = np.mean(abs(refit_res1['m_obs1'] - m_true))
                refit_obs_std1 = np.std(abs(refit_res1['m_obs1'] - m_true))
                refit_obs_ste1 = refit_obs_std1/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, Observed)', refit_obs1, refit_obs_std1, refit_obs_ste1]
                
                refit_ipw1 = np.mean(abs(refit_res1['m_ipw1'] - m_true))
                refit_ipw_std1 = np.std(abs(refit_res1['m_ipw1'] - m_true))
                refit_ipw_ste1 = refit_ipw_std1/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, IPW)', refit_ipw1, refit_ipw_std1, refit_ipw_ste1]
                
                refit_obs2 = np.mean(abs(refit_res1['m_obs2'] - m_true))
                refit_obs_std2 = np.std(abs(refit_res1['m_obs2'] - m_true))
                refit_obs_ste2 = refit_obs_std2/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, Observed)', refit_obs2, refit_obs_std2, refit_obs_ste2]
                
                refit_ipw2 = np.mean(abs(refit_res1['m_ipw2'] - m_true))
                refit_ipw_std2 = np.std(abs(refit_res1['m_ipw2'] - m_true))
                refit_ipw_ste2 = refit_ipw_std2/np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, IPW)', refit_ipw2, refit_ipw_std2, refit_ipw_ste2]
                
                refit_full = np.mean(abs(refit_res1['m_full'] - m_true))
                refit_full_std = np.std(abs(refit_res1['m_full'] - m_true))
                refit_full_ste = refit_full_std / np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (Full oracle data)', refit_full, refit_full_std, refit_full_ste]
                
                ## Our proposed debiasing framework
                m1 = np.mean(abs(deb_est_mincv['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_mincv['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, Mincv)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(deb_est_mincv['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_mincv['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, Mincv)', m2, m_std2, m_ste2]
                
                m1 = np.mean(abs(deb_est_1se['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_1se['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, 1SE)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(deb_est_1se['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_1se['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, 1SE)', m2, m_std2, m_ste2]
                
                m1 = np.mean(abs(deb_est_minfeas['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_minfeas['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MCAR, Min-feas)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                Bias1.loc[len(Bias1.index)] = ['Proposed framework (MAR, Min-feas)', m2, m_std2, m_ste2]

                ## Tian2024 AIPW
                m1 = np.mean(abs(tian_res['m_hat1'] - m_true))
                m_std1 = np.std(abs(tian_res['m_hat1'] - m_true))
                m_ste1 = m_std1 / np.sqrt(len(tian_res))
                Bias1.loc[len(Bias1.index)] = ['Tian2024 AIPW (MCAR)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(tian_res['m_hat2'] - m_true))
                m_std2 = np.std(abs(tian_res['m_hat2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(len(tian_res))
                Bias1.loc[len(Bias1.index)] = ['Tian2024 AIPW (MAR)', m2, m_std2, m_ste2]
                
                ## Chakrabortty et al. (2019) DDR
                m1 = np.mean(abs(ddr_res['m_hat_de1'] - m_true))
                m_std1 = np.std(abs(ddr_res['m_hat_de1'] - m_true))
                m_ste1 = m_std1 / np.sqrt(len(ddr_res))
                Bias1.loc[len(Bias1.index)] = ['DDR (MCAR)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(ddr_res['m_hat_de2'] - m_true))
                m_std2 = np.std(abs(ddr_res['m_hat_de2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(len(ddr_res))
                Bias1.loc[len(Bias1.index)] = ['DDR (MAR)', m2, m_std2, m_ste2]
                
                ## Hou et al. (2023) debiased SAS
                m1 = np.mean(abs(hou_sas_res['m_hat1'] - m_true))
                m_std1 = np.std(abs(hou_sas_res['m_hat1'] - m_true))
                m_ste1 = m_std1 / np.sqrt(len(hou_sas_res))
                Bias1.loc[len(Bias1.index)] = ['Hou2023 SAS (MCAR)', m1, m_std1, m_ste1]
                
                m2 = np.mean(abs(hou_sas_res['m_hat2'] - m_true))
                m_std2 = np.std(abs(hou_sas_res['m_hat2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(len(hou_sas_res))
                Bias1.loc[len(Bias1.index)] = ['Hou2023 SAS (MAR)', m2, m_std2, m_ste2]
            
            
                # Coverage Probability
                ## Debiased lasso (Javanmard and Montarani, 2014)
                cov_prob = (debl_res1['m_obs1'] + debl_res1['ci_len_obs1']/2 >= m_true) & \
                (debl_res1['m_obs1'] - debl_res1['ci_len_obs1']/2 <= m_true)
                # Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
                Cov_prob1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                
                cov_prob = (debl_res1['m_ipw1'] + debl_res1['ci_len_ipw1']/2 >= m_true) & \
                (debl_res1['m_ipw1'] - debl_res1['ci_len_ipw1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (debl_res1['m_obs2'] + debl_res1['ci_len_obs2']/2 >= m_true) & \
                (debl_res1['m_obs2'] - debl_res1['ci_len_obs2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (debl_res1['m_ipw2'] + debl_res1['ci_len_ipw2']/2 >= m_true) & \
                (debl_res1['m_ipw2'] - debl_res1['ci_len_ipw2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (debl_res1['m_full'] + debl_res1['ci_len_full']/2 >= m_true) & \
                (debl_res1['m_full'] - debl_res1['ci_len_full']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (Javanmard) (Full data)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                ## Debiased lasso (van de geer et al., 2014)
                cov_prob = (lproj_res1['m_obs1'] + np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (lproj_res1['m_obs1'] - np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
    
                cov_prob = (lproj_res1['m_ipw1'] + np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (lproj_res1['m_ipw1'] - np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
    
                cov_prob = (lproj_res1['m_obs2'] + np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (lproj_res1['m_obs2'] - np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
    
                cov_prob = (lproj_res1['m_ipw2'] + np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (lproj_res1['m_ipw2'] - np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
    
                cov_prob = (lproj_res1['m_full'] + np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (lproj_res1['m_full'] - np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Debiased lasso (van de geer) (MAR, Full data)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                ## Ridge projection
                if (i == 0) or (i == 2):
                    cov_prob = (rproj_res1['m_obs1'] + rproj_res1['ci_len_obs1']/2 >= m_true) & \
                    (rproj_res1['m_obs1'] - rproj_res1['ci_len_obs1']/2 <= m_true)
                    Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, Observed)', np.mean(cov_prob), 
                                                           np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                    
                    cov_prob = (rproj_res1['m_ipw1'] + rproj_res1['ci_len_ipw1']/2 >= m_true) & \
                    (rproj_res1['m_ipw1'] - rproj_res1['ci_len_ipw1']/2 <= m_true)
                    Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(cov_prob), 
                                                           np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                    
                    cov_prob = (rproj_res1['m_obs2'] + rproj_res1['ci_len_obs2']/2 >= m_true) & \
                    (rproj_res1['m_obs2'] - rproj_res1['ci_len_obs2']/2 <= m_true)
                    Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, Observed)', np.mean(cov_prob), 
                                                           np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                    
                    cov_prob = (rproj_res1['m_ipw2'] + rproj_res1['ci_len_ipw2']/2 >= m_true) & \
                    (rproj_res1['m_ipw2'] - rproj_res1['ci_len_ipw2']/2 <= m_true)
                    Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (MAR, IPW)', np.mean(cov_prob), 
                                                           np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                    
                    cov_prob = (rproj_res1['m_full'] + rproj_res1['ci_len_full']/2 >= m_true) & \
                    (rproj_res1['m_full'] - rproj_res1['ci_len_full']/2 <= m_true)
                    Cov_prob1.loc[len(Cov_prob1.index)] = ['Ridge projection (Full data)', np.mean(cov_prob), 
                                                           np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                ## Lasso refitting
                cov_prob = (refit_res1['m_obs1'] + refit_res1['ci_len_obs1']/2 >= m_true) & \
                (refit_res1['m_obs1'] - refit_res1['ci_len_obs1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (refit_res1['m_ipw1'] + refit_res1['ci_len_ipw1']/2 >= m_true) & \
                (refit_res1['m_ipw1'] - refit_res1['ci_len_ipw1']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (refit_res1['m_obs2'] + refit_res1['ci_len_obs2']/2 >= m_true) & \
                (refit_res1['m_obs2'] - refit_res1['ci_len_obs2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (refit_res1['m_ipw2'] + refit_res1['ci_len_ipw2']/2 >= m_true) & \
                (refit_res1['m_ipw2'] - refit_res1['ci_len_ipw2']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (refit_res1['m_full'] + refit_res1['ci_len_full']/2 >= m_true) & \
                (refit_res1['m_full'] - refit_res1['ci_len_full']/2 <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (Full data)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                # Proposed framework   
                cov_prob = (deb_est_mincv['m_deb1'] + deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_mincv['m_deb1'] - deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, Mincv)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (deb_est_mincv['m_deb2'] + deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_mincv['m_deb2'] - deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, Mincv)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (deb_est_1se['m_deb1'] + deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_1se['m_deb1'] - deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, 1SE)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (deb_est_1se['m_deb2'] + deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_1se['m_deb2'] - deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, 1SE)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (deb_est_minfeas['m_deb1'] + deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_minfeas['m_deb1'] - deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MCAR, Min-feas)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                cov_prob = (deb_est_minfeas['m_deb2'] + deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_minfeas['m_deb2'] - deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Proposed framework (MAR, Min-feas)', np.mean(cov_prob), 
                                                   np.std(cov_prob), np.std(cov_prob)/np.sqrt(1000)]
                
                ## Tian2024 AIPW
                cov_prob = (tian_res['m_hat1'] + tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (tian_res['m_hat1'] - tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Tian2024 AIPW (MCAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                cov_prob = (tian_res['m_hat2'] + tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (tian_res['m_hat2'] - tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Tian2024 AIPW (MAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                ## Chakrabortty et al. (2019) DDR
                cov_prob = (ddr_res['ci_lower1'] <= m_true) & (m_true <= ddr_res['ci_upper1'])
                Cov_prob1.loc[len(Cov_prob1.index)] = ['DDR (MCAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                cov_prob = (ddr_res['ci_lower2'] <= m_true) & (m_true <= ddr_res['ci_upper2'])
                Cov_prob1.loc[len(Cov_prob1.index)] = ['DDR (MAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                ## Hou et al. (2023) SAS
                cov_prob = (hou_sas_res['m_hat1'] + hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (hou_sas_res['m_hat1'] - hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Hou2023 SAS (MCAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
                
                cov_prob = (hou_sas_res['m_hat2'] + hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (hou_sas_res['m_hat2'] - hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                Cov_prob1.loc[len(Cov_prob1.index)] = ['Hou2023 SAS (MAR)', np.mean(cov_prob), 
                                                       np.std(cov_prob), np.std(cov_prob)/np.sqrt(len(cov_prob))]
        
            
                # Lengths of confidence intervals
                ## Debiased lasso (Javanmard and Montarani, 2014)
                CI_len1 = pd.DataFrame({'Scenario': ['Debiased lasso (Javanmard) (MCAR, Observed)'], 
                                'CI length': [np.mean(debl_res1['ci_len_obs1'])], 
                                'Std': [np.std(debl_res1['ci_len_obs1'])], 
                                'StdErr': [np.std(debl_res1['ci_len_obs1'])/np.sqrt(1000)]})
    #             CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, Observed)', np.mean(debl_res1['ci_len_obs1']), 
    #                                                np.std(debl_res1['ci_len_obs1'])]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MCAR, IPW)', np.mean(debl_res1['ci_len_ipw1']), 
                                                   np.std(debl_res1['ci_len_ipw1']), 
                                                   np.std(debl_res1['ci_len_ipw1'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, Observed)', np.mean(debl_res1['ci_len_obs2']), 
                                                   np.std(debl_res1['ci_len_obs2']), 
                                                   np.std(debl_res1['ci_len_obs2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (MAR, IPW)', np.mean(debl_res1['ci_len_ipw2']), 
                                                   np.std(debl_res1['ci_len_ipw2']), 
                                                   np.std(debl_res1['ci_len_ipw2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (Javanmard) (Full oracle data)', np.mean(debl_res1['ci_len_full']), 
                                                   np.std(debl_res1['ci_len_full']), np.std(debl_res1['ci_len_full'])/np.sqrt(1000)]
                
                ## Debiased lasso (van de geer et al., 2014)
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, Observed)', 
                                                   np.mean(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_obs1'])*lproj_res1['sigma_hat_obs1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MCAR, IPW)', 
                                                   np.mean(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_ipw1'])*lproj_res1['sigma_hat_ipw1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, Observed)', 
                                                   np.mean(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_obs2'])*lproj_res1['sigma_hat_obs2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (MAR, IPW)', 
                                                   np.mean(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_ipw2'])*lproj_res1['sigma_hat_ipw2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
                CI_len1.loc[len(CI_len1.index)] = ['Debiased lasso (van de geer) (Full oracle data)', 
                                                   np.mean(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*np.sqrt(lproj_res1['asym_var_full'])*lproj_res1['sigma_hat_full']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(lproj_res1))]
                
                ## Ridge projection
                if (i == 0) or (i == 2):
                    CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, Observed)', 
                                                       np.mean(rproj_res1['ci_len_obs1']), 
                                                       np.std(rproj_res1['ci_len_obs1']), 
                                                       np.std(rproj_res1['ci_len_obs1'])/np.sqrt(1000)]
                    CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MCAR, IPW)', np.mean(rproj_res1['ci_len_ipw1']), 
                                                       np.std(rproj_res1['ci_len_ipw1']), 
                                                       np.std(rproj_res1['ci_len_ipw1'])/np.sqrt(1000)]
                    CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, Observed)', np.mean(rproj_res1['ci_len_obs2']), 
                                                       np.std(rproj_res1['ci_len_obs2']), 
                                                       np.std(rproj_res1['ci_len_obs2'])/np.sqrt(1000)]
                    CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (MAR, IPW)', np.mean(rproj_res1['ci_len_ipw2']), 
                                                       np.std(rproj_res1['ci_len_ipw2']), 
                                                       np.std(rproj_res1['ci_len_ipw2'])/np.sqrt(1000)]
                    CI_len1.loc[len(CI_len1.index)] = ['Ridge projection (Full oracle data)', np.mean(rproj_res1['ci_len_full']), 
                                                       np.std(rproj_res1['ci_len_full']), 
                                                       np.std(rproj_res1['ci_len_full'])/np.sqrt(1000)]
                
                ## Lasso refitting
                CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, Observed)', np.mean(refit_res1['ci_len_obs1']), 
                                                   np.std(refit_res1['ci_len_obs1']), 
                                                   np.std(refit_res1['ci_len_obs1'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(refit_res1['ci_len_ipw1']), 
                                                   np.std(refit_res1['ci_len_ipw1']), 
                                                   np.std(refit_res1['ci_len_ipw1'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(refit_res1['ci_len_obs2']), 
                                                   np.std(refit_res1['ci_len_obs2']), 
                                                   np.std(refit_res1['ci_len_obs2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(refit_res1['ci_len_ipw2']), 
                                                   np.std(refit_res1['ci_len_ipw2']), 
                                                   np.std(refit_res1['ci_len_ipw2'])/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (Full oracle data)', np.mean(refit_res1['ci_len_full']), 
                                                   np.std(refit_res1['ci_len_full']), 
                                                   np.std(refit_res1['ci_len_full'])/np.sqrt(1000)]
                
                ## Proposed debiasing framework
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, Mincv)', 
                                                   np.mean(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, Mincv)', 
                                                   np.mean(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, 1SE)', 
                                                   np.mean(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, 1SE)', 
                                                   np.mean(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MCAR, Min-feas)', 
                                                   np.mean(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                CI_len1.loc[len(CI_len1.index)] = ['Proposed framework (MAR, Min-feas)', 
                                                   np.mean(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]
                
                ## Tian2024 AIPW
                CI_len1.loc[len(CI_len1.index)] = ['Tian2024 AIPW (MCAR)', 
                                                   np.mean(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*tian_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(tian_res))]
                CI_len1.loc[len(CI_len1.index)] = ['Tian2024 AIPW (MAR)', 
                                                   np.mean(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*tian_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(tian_res))]
                
                ## Chakrabortty et al. (2019) DDR
                CI_len1.loc[len(CI_len1.index)] = ['DDR (MCAR)', np.mean(ddr_res['ci_length1']), 
                                                   np.std(ddr_res['ci_length1']), 
                                                   np.std(ddr_res['ci_length1'])/np.sqrt(len(ddr_res))]
                CI_len1.loc[len(CI_len1.index)] = ['DDR (MAR)', np.mean(ddr_res['ci_length2']), 
                                                   np.std(ddr_res['ci_length2']), 
                                                   np.std(ddr_res['ci_length2'])/np.sqrt(len(ddr_res))]
                
                ## Hou et al. (2023) SAS
                CI_len1.loc[len(CI_len1.index)] = ['Hou2023 SAS (MCAR)', 
                                                   np.mean(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*hou_sas_res['asym_var1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(hou_sas_res))]
                CI_len1.loc[len(CI_len1.index)] = ['Hou2023 SAS (MAR)', 
                                                   np.mean(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2)), 
                                                   np.std(2*hou_sas_res['asym_var2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(len(hou_sas_res))]


                full_res1 = pd.concat([Bias1, Cov_prob1[['Coverage Probability', 'Std', 'StdErr']], 
                                       CI_len1[['CI length', 'Std', 'StdErr']]], axis=1)
                
                # full_res1.to_csv('./Results/Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'.csv', index=False)
                full_res1.to_excel(writer, sheet_name='Cirsym_cov_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k), index=False)

### d=1000, n=900 (Nonparametric propensity score estimation)

In [3]:
d = 1000
n = 900
with pd.ExcelWriter('./Results/Cirsym_Nonpar_Prop_Full_Results_d'+str(d)+'_n'+str(n)+'.xlsx') as writer:
    for i in [0,1,2,4]:
        if i == 0:
            ## x0
            x = np.zeros((d,))
            x[0] = 1
        if i == 1:
            ## x1
            x = np.zeros((d,))
            x[0] = 1
            x[1] = 1/2
            x[2] = 1/4
            x[6] = 1/2
            x[7] = 1/8
        if i == 2:
            ## x2
            x = np.zeros((d,))
            x[99] = 1
        if i == 3:
            ## x3
            x = 1/np.linspace(1, d, d)
        if i == 4:
            ## x4
            x = 1/np.linspace(1, d, d)**2
        for k in [0,2]:
            if k == 0:
                s_beta = 5
                beta_0 = np.zeros((d,))
                beta_0[:s_beta] = np.sqrt(5)
            if k == 1:
                beta_0 = 1/np.sqrt(np.linspace(1, d, d))
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
            if k == 2:
                beta_0 = 1/np.linspace(1, d, d)
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
    
            # True regression function
            m_true = np.dot(x, beta_0)
            
            Bias2 = pd.DataFrame([])
            Cov_prob2 = pd.DataFrame([])
            CI_len2 = pd.DataFrame([])
            ## Propensity score estimation (Naive Bayes, Random Forests, SVM, MLP neural network)
            for non_met in ['NB', 'NBcal', 'RF', 'RFcal', 'SVM', 'SVMcal', 'NN', 'NNcal']:
                deb_est_1se = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_1se.csv')
                deb_est_mincv = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_mincv.csv')
                deb_est_minfeas = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_minfeas.csv')
                
                # Bias comparison
                m1 = np.mean(abs(deb_est_mincv['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_mincv['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-CV)'], 
                                        'Avg bias': [m1], 'Std': [m_std1], 'StdErr': [m_ste1]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                m1 = np.mean(abs(deb_est_1se['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_1se['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, 1SE)'], 
                                        'Avg bias': [m1], 'Std': [m_std1], 'StdErr': [m_ste1]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                m1 = np.mean(abs(deb_est_minfeas['m_deb1'] - m_true))
                m_std1 = np.std(abs(deb_est_minfeas['m_deb1'] - m_true))
                m_ste1 = m_std1 /np.sqrt(1000)
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-feas)'], 
                                        'Avg bias': [m1], 'Std': [m_std1], 'StdErr': [m_ste1]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                
                m2 = np.mean(abs(deb_est_mincv['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_mincv['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                m2 = np.mean(abs(deb_est_1se['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_1se['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                
                m2 = np.mean(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-feas)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                ## Coverage probability
                cov_prob = (deb_est_mincv['m_deb1'] + deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_mincv['m_deb1'] - deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-CV)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                
                cov_prob = (deb_est_1se['m_deb1'] + deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_1se['m_deb1'] - deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, 1SE)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_minfeas['m_deb1'] + deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_minfeas['m_deb1'] - deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-feas)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_mincv['m_deb2'] + deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_mincv['m_deb2'] - deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_1se['m_deb2'] + deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_1se['m_deb2'] - deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_minfeas['m_deb2'] + deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_minfeas['m_deb2'] - deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-feas)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                ## CI Lengths
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-CV)'], 
                                'CI length': [np.mean(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_mincv['asym_se1']*deb_est_mincv['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, 1SE)'], 
                                'CI length': [np.mean(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_1se['asym_se1']*deb_est_1se['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-feas)'], 
                                'CI length': [np.mean(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_minfeas['asym_se1']*deb_est_minfeas['sigma_hat1']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                'CI length': [np.mean(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                'CI length': [np.mean(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-feas)'], 
                                'CI length': [np.mean(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
            full_res2 = pd.concat([Bias2, Cov_prob2[['Coverage Probability', 'Std', 'StdErr']], 
                                   CI_len2[['CI length', 'Std', 'StdErr']]], axis=1)
            
            full_res2.to_excel(writer, sheet_name='Cirsym_cov_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k), index=False)

### d=1000, n=900 (Nonparametric Propensity Score Estimation with a Misspecificed Propensity Score Model)

In [8]:
d = 1000
n = 900
with pd.ExcelWriter('./Results/Cirsym_Nonpar_Prop_Misscore_Full_Results_d'+str(d)+'_n'+str(n)+'_mis.xlsx') as writer:
    for i in [0,1,2,4]:
        if i == 0:
            ## x0
            x = np.zeros((d,))
            x[0] = 1
        if i == 1:
            ## x1
            x = np.zeros((d,))
            x[0] = 1
            x[1] = 1/2
            x[2] = 1/4
            x[6] = 1/2
            x[7] = 1/8
        if i == 2:
            ## x2
            x = np.zeros((d,))
            x[99] = 1
        if i == 3:
            ## x3
            x = 1/np.linspace(1, d, d)
        if i == 4:
            ## x4
            x = 1/np.linspace(1, d, d)**2
        for k in [0,1,2]:
            if k == 0:
                s_beta = 5
                beta_0 = np.zeros((d,))
                beta_0[:s_beta] = np.sqrt(5)
            if k == 1:
                beta_0 = 1/np.sqrt(np.linspace(1, d, d))
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
            if k == 2:
                beta_0 = 1/np.linspace(1, d, d)
                beta_0 = 5*beta_0/np.linalg.norm(beta_0)
    
            # True regression function
            m_true = np.dot(x, beta_0)
            
            Bias2 = pd.DataFrame([])
            Cov_prob2 = pd.DataFrame([])
            CI_len2 = pd.DataFrame([])
            MAE_prop2 = pd.DataFrame([])
            ## Propensity score estimation (Naive Bayes, Random Forests, SVM, MLP neural network)
            for non_met in ['Oracle', 'LR', 'NB', 'NBcal', 'RF', 'RFcal', 'SVM', 'SVMcal', 'NN', 'NNcal']:
                deb_est_1se = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_1se_mis.csv')
                deb_est_mincv = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_mincv_mis.csv')
                deb_est_minfeas = pd.read_csv('./Results/DebiasProg_Cirsym_cov_homoerr_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k)+'_prop_'+str(non_met)+'_minfeas_mis.csv')
                
                # Bias comparison
                
                m2 = np.mean(abs(deb_est_mincv['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_mincv['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                m2 = np.mean(abs(deb_est_1se['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_1se['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                
                m2 = np.mean(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_std2 = np.std(abs(deb_est_minfeas['m_deb2'] - m_true))
                m_ste2 = m_std2 / np.sqrt(1000) 
                bias_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-feas)'], 
                                        'Avg bias': [m2], 'Std': [m_std2], 'StdErr': [m_ste2]})
                Bias2 = pd.concat([Bias2, bias_df], axis=0)
                
                ## Coverage probability
                
                cov_prob = (deb_est_mincv['m_deb2'] + deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_mincv['m_deb2'] - deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_1se['m_deb2'] + deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_1se['m_deb2'] - deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                cov_prob = (deb_est_minfeas['m_deb2'] + deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) >= m_true) & \
                (deb_est_minfeas['m_deb2'] - deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2) <= m_true)
                cov_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-feas)'], 
                                          'Coverage Probability': [np.mean(cov_prob)], 
                                          'Std': [np.std(cov_prob)], 
                                          'StdErr': [np.std(cov_prob)/np.sqrt(1000)]})
                Cov_prob2 = pd.concat([Cov_prob2, cov_df], axis=0)
                
                ## CI Lengths
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                'CI length': [np.mean(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_mincv['asym_se2']*deb_est_mincv['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
            
                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                'CI length': [np.mean(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_1se['asym_se2']*deb_est_1se['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)

                
                CI_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MCAR, min-feas)'], 
                                'CI length': [np.mean(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'Std': [np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))], 
                                'StdErr': [np.std(2*deb_est_minfeas['asym_se2']*deb_est_minfeas['sigma_hat2']*scipy.stats.norm.ppf(1-0.05/2))/np.sqrt(1000)]})
                CI_len2 = pd.concat([CI_len2, CI_df], axis=0)
                
                
                ## MAE for propensity score
                mae_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-CV)'], 
                                       'Average Prop Est MAE': [np.mean(deb_est_mincv['mae_prop'])], 
                                        'Std': [np.std(deb_est_mincv['mae_prop'])], 
                                        'StdErr': [np.std(deb_est_mincv['mae_prop'])/np.sqrt(1000)]})
                MAE_prop2 = pd.concat([MAE_prop2, mae_df], axis=0)
                
                mae_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, 1SE)'], 
                                       'Average Prop Est MAE': [np.mean(deb_est_1se['mae_prop'])], 
                                        'Std': [np.std(deb_est_1se['mae_prop'])], 
                                        'StdErr': [np.std(deb_est_1se['mae_prop'])/np.sqrt(1000)]})
                MAE_prop2 = pd.concat([MAE_prop2, mae_df], axis=0)
                
                mae_df = pd.DataFrame({'Scenario': ['Proposed framework ('+non_met+', MAR, min-feas)'], 
                                       'Average Prop Est MAE': [np.mean(deb_est_minfeas['mae_prop'])], 
                                        'Std': [np.std(deb_est_minfeas['mae_prop'])], 
                                        'StdErr': [np.std(deb_est_minfeas['mae_prop'])/np.sqrt(1000)]})
                MAE_prop2 = pd.concat([MAE_prop2, mae_df], axis=0)
                
            full_res2 = pd.concat([Bias2, Cov_prob2[['Coverage Probability', 'Std', 'StdErr']], 
                                   CI_len2[['CI length', 'Std', 'StdErr']], 
                                   MAE_prop2[['Average Prop Est MAE', 'Std', 'StdErr']]], axis=1)
            
            full_res2.to_excel(writer, sheet_name='Cirsym_cov_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta'+str(k), index=False)

### Lasso Refitting (Increasing the noise levels)

In [2]:
## Homoscedastic case
d = 1000
n = 900
with pd.ExcelWriter('./Results/Lasso_Refit_Cirsym_Full_Results_d'+str(d)+'_n'+str(n)+'.xlsx') as writer:
    for i in range(4):
        if i == 0:
            ## x0
            x = np.zeros((d,))
            x[0] = 1
        if i == 1:
            ## x1
            x = np.zeros((d,))
            x[0] = 1
            x[1] = 1/2
            x[2] = 1/4
            x[6] = 1/2
            x[7] = 1/8
        if i == 2:
            ## x2
            x = np.zeros((d,))
            x[99] = 1
        if i == 3:
            ## x3
            x = 1/np.linspace(1, d, d)**2
            
        k = 0
        s_beta = 5
        beta_0 = np.zeros((d,))
        beta_0[:s_beta] = np.sqrt(5)
        
        for sig in [1,2,3,4,5]:
            # True regression function
            m_true = np.dot(x, beta_0)
            refit_res1 = pd.read_csv('./Results/refit_Cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta0_sig_'+str(sig)+'.csv')
            
            ## Lasso Refitting
            refit_obs1 = np.mean(abs(refit_res1['m_obs1'] - m_true))
            refit_obs_std1 = np.std(abs(refit_res1['m_obs1'] - m_true))
            Bias1 = pd.DataFrame({'Scenario': ['Lasso Refitting (MCAR, Observed)'], 
                                  'Avg bias': [refit_obs1], 'Std': [refit_obs_std1]})
            
            refit_ipw1 = np.mean(abs(refit_res1['m_ipw1'] - m_true))
            refit_ipw_std1 = np.std(abs(refit_res1['m_ipw1'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MCAR, IPW)', refit_ipw1, refit_ipw_std1]
            
            refit_obs2 = np.mean(abs(refit_res1['m_obs2'] - m_true))
            refit_obs_std2 = np.std(abs(refit_res1['m_obs2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, Observed)', refit_obs2, refit_obs_std2]
            
            refit_ipw2 = np.mean(abs(refit_res1['m_ipw2'] - m_true))
            refit_ipw_std2 = np.std(abs(refit_res1['m_ipw2'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (MAR, IPW)', refit_ipw2, refit_ipw_std2]
            
            refit_full = np.mean(abs(refit_res1['m_full'] - m_true))
            refit_full_std = np.std(abs(refit_res1['m_full'] - m_true))
            Bias1.loc[len(Bias1.index)] = ['Lasso Refitting (Full oracle data)', refit_full, refit_full_std]
            
            
            ## Lasso refitting
            cov_prob = (refit_res1['m_obs1'] + refit_res1['ci_len_obs1']/2 >= m_true) & \
            (refit_res1['m_obs1'] - refit_res1['ci_len_obs1']/2 <= m_true)
            Cov_prob1 = pd.DataFrame({'Scenario': ['Lasso refitting (MCAR, Observed)'], 
                                      'Coverage Probability': [np.mean(cov_prob)], 
                                      'Std': [np.std(cov_prob)]})
            
            cov_prob = (refit_res1['m_ipw1'] + refit_res1['ci_len_ipw1']/2 >= m_true) & \
            (refit_res1['m_ipw1'] - refit_res1['ci_len_ipw1']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_obs2'] + refit_res1['ci_len_obs2']/2 >= m_true) & \
            (refit_res1['m_obs2'] - refit_res1['ci_len_obs2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_ipw2'] + refit_res1['ci_len_ipw2']/2 >= m_true) & \
            (refit_res1['m_ipw2'] - refit_res1['ci_len_ipw2']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(cov_prob), np.std(cov_prob)]
            
            cov_prob = (refit_res1['m_full'] + refit_res1['ci_len_full']/2 >= m_true) & \
            (refit_res1['m_full'] - refit_res1['ci_len_full']/2 <= m_true)
            Cov_prob1.loc[len(Cov_prob1.index)] = ['Lasso refitting (Full data)', np.mean(cov_prob), np.std(cov_prob)]
            
            
            ## Lasso refitting
            CI_len1 = pd.DataFrame({'Scenario': ['Lasso refitting (MCAR, Observed)'], 
                            'CI length': [np.mean(refit_res1['ci_len_obs1'])], 
                            'Std': [np.std(refit_res1['ci_len_obs1'])]})
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MCAR, IPW)', np.mean(refit_res1['ci_len_ipw1']), 
                                               np.std(refit_res1['ci_len_ipw1'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, Observed)', np.mean(refit_res1['ci_len_obs2']), 
                                               np.std(refit_res1['ci_len_obs2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (MAR, IPW)', np.mean(refit_res1['ci_len_ipw2']), 
                                               np.std(refit_res1['ci_len_ipw2'])]
            CI_len1.loc[len(CI_len1.index)] = ['Lasso refitting (Full oracle data)', np.mean(refit_res1['ci_len_full']), 
                                               np.std(refit_res1['ci_len_full'])]
            
            full_res1 = pd.concat([Bias1, Cov_prob1[['Coverage Probability', 'Std']], CI_len1[['CI length', 'Std']]], axis=1)
            
            full_res1.to_excel(writer, sheet_name='Cirsym_d'+str(d)+'_n'+str(n)+'_x'+str(i)+'_beta0_sig_'+str(sig), index=False)
            

/usr/local/lib/python3.10/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/usr/local/lib/python3.10/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/usr/local/lib/python3.10/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/usr/local/lib/python3.10/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.w

In [8]:
lproj_res1

,m_obs1,asym_var_obs1,sigma_hat_obs1,ci_len_obs1,m_obs2,asym_var_obs2,sigma_hat_obs2,ci_len_obs2,m_ipw1,asym_var_ipw1,sigma_hat_ipw1,ci_len_ipw1,m_ipw2,asym_var_ipw2,sigma_hat_ipw2,ci_len_ipw2,m_full,asym_var_full,sigma_hat_full,ci_len_full
0,4.803723,0.001901,2.664214,0.455323,4.722626,0.001912,2.366994,0.405716,4.759278,0.001331,2.900579,0.414748,4.608137,0.001320,2.565511,0.365362,4.744664,0.001259,2.532005,0.352232
1,4.644566,0.001860,2.313099,0.391064,4.839052,0.001861,3.171762,0.536367,4.614044,0.001302,2.603427,0.368255,4.800363,0.001301,3.518293,0.497520,4.724290,0.001284,2.927522,0.411254
2,5.128856,0.001963,3.220257,0.559313,5.422940,0.002032,4.657904,0.823135,5.076258,0.001374,3.627358,0.527113,5.221004,0.001372,4.923487,0.714865,5.200518,0.001371,4.142611,0.601335
3,4.826935,0.001974,2.791001,0.486074,4.860839,0.002080,3.070356,0.548958,4.772890,0.001382,3.115225,0.453922,4.752562,0.001347,3.249786,0.467587,4.782989,0.001374,2.822636,0.410068
4,4.638456,0.001869,3.658866,0.620078,4.693590,0.001849,3.574181,0.602405,4.587664,0.001308,4.183905,0.593241,4.640196,0.001272,3.939681,0.550739,4.681555,0.001258,3.291360,0.457552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4.721565,0.001936,2.361422,0.407243,4.710073,0.002053,2.406137,0.427363,4.688433,0.001355,2.615062,0.377321,4.695187,0.001404,2.633198,0.386746,4.673339,0.001351,2.678470,0.385900
996,4.766569,0.001753,2.594656,0.425887,4.828760,0.001852,2.563951,0.432521,4.732009,0.001227,2.897410,0.397900,4.780013,0.001369,2.798642,0.405942,4.726848,0.001262,2.452087,0.341421
997,4.714417,0.001741,3.991719,0.652856,4.691657,0.001835,3.972484,0.666996,4.671684,0.001219,4.567003,0.624940,4.614169,0.001235,4.598375,0.633512,4.647470,0.001247,3.637699,0.503466
998,4.933854,0.001748,4.412901,0.723250,5.152730,0.001709,6.193958,1.003867,4.880649,0.001224,5.041124,0.691259,5.007169,0.001152,6.836554,0.909666,5.007177,0.001160,5.825485,0.777813
